In [1]:
import requests, json, time

OLLAMA_MODEL = "qwen3.5:4b"

def call_llm(prompt, model=OLLAMA_MODEL, temperature=0.6, max_tokens=800):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False, "think": False,
                  "options": {"temperature": temperature, "num_predict": max_tokens}},
            timeout=90
        )
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"LLM call failed: {e}"

print("LLM helper loaded")

LLM helper loaded


In [2]:
with open("../outputs/resume_text.txt", "r", encoding="utf-8") as f:
    resume_text = f.read()

with open("../outputs/skill_gap.json", "r", encoding="utf-8") as f:
    gap_data = json.load(f)

with open("../outputs/resume_improvements.json", "r", encoding="utf-8") as f:
    improvements = json.load(f)

target_role = gap_data["target_role"]
print(f"Loaded resume data. Target role: {target_role}")

Loaded resume data. Target role: AI Engineer


In [3]:
def craft_improved_resume(original_text, target_role, missing_skills, improvement_feedback):
    prompt = f"""You are an expert resume writer. Rewrite and improve this resume to better target a {target_role} role.

Rules:
- Keep all factual information (name, contact, real education, real projects) unchanged
- Improve wording, add quantifiable impact where reasonable, reorganize for clarity
- Naturally incorporate mentions of these skills ONLY if plausible given their existing projects: {', '.join(missing_skills[:4])}
- Do not fabricate false experience or credentials
- Output in clean plain text resume format with clear section headers

IMPROVEMENT NOTES TO APPLY:
{improvement_feedback[:800]}

ORIGINAL RESUME:
{original_text[:2000]}

Output the improved resume text only, no commentary."""

    return call_llm(prompt, temperature=0.5, max_tokens=900)

start = time.time()
crafted_resume = craft_improved_resume(
    resume_text, target_role, gap_data["missing_skills"],
    improvements.get("llm_generated_feedback", "")
)
elapsed = time.time() - start

print(f"Resume Crafting Agent: SUCCESS (generated in {elapsed:.2f}s)\n")
print(crafted_resume)

Resume Crafting Agent: SUCCESS (generated in 14.58s)

ASHISH KUMAR
STUDENT
RR Line Enclave, Ambala, Haryana | 9518822674 | ashisaini18@gmail.com

ABOUT ME
B.Sc. (AI & ML) student at Maharishi Markandeshwar University with a strong foundation in Artificial Intelligence, Data Science, and Full-Stack Web Development. Proficient in Python, Java, C/C++, SQL, HTML, CSS, and JavaScript, with specialized expertise in NumPy, Pandas, TensorFlow, PyTorch, Flask, FastAPI, Docker, and Kubernetes concepts. Passionate about building scalable intelligent solutions that solve real-world business problems, exemplified by the "Predictive Supply Chain AI" project which anticipates disruptions before they occur. Strong analytical mindset focused on model optimization, system deployment, and continuous learning in modern machine engineering stacks.

EDUCATION
2024 - 2027 | Bachelor of Science (AI & ML)
Maharishi Markandeshwar University, Ambala
CGPA: 8.4/10 (3rd Semester)
Key Focus Areas: Deep Learning, Mac

In [4]:
with open("../outputs/crafted_resume.txt", "w", encoding="utf-8") as f:
    f.write(crafted_resume)

print("Crafted resume saved to ../outputs/crafted_resume.txt")
print("Notebook 14 (Resume Crafting Agent) — COMPLETE")

Crafted resume saved to ../outputs/crafted_resume.txt
Notebook 14 (Resume Crafting Agent) — COMPLETE


In [5]:
import re as regex_module

def calculate_ats_score_simple(text):
    score = 0
    feedback = []

    has_email = bool(regex_module.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', text))
    has_phone = bool(regex_module.search(r'\d{10}', text))
    if has_email: score += 10
    else: feedback.append("Add email")
    if has_phone: score += 10
    else: feedback.append("Add phone")

    TECHNICAL_SKILLS = [
        "Python", "Java", "JavaScript", "C++", "C", "SQL", "HTML", "CSS",
        "Flask", "FastAPI", "Docker", "Kubernetes", "Git", "Machine Learning",
        "TensorFlow", "PyTorch", "NumPy", "Pandas", "Deep Learning"
    ]
    text_lower = text.lower()
    skills_found = [s for s in TECHNICAL_SKILLS if s.lower() in text_lower]
    skill_count = len(skills_found)
    if skill_count >= 8: score += 30
    elif skill_count >= 5: score += 20
    elif skill_count >= 1: score += 10

    section_keywords = ["experience", "education", "project", "skill", "certification", "achievement"]
    found_sections = [kw for kw in section_keywords if kw in text_lower]
    score += min(len(found_sections) * 5, 30)

    word_count = len(text.split())
    if word_count >= 150: score += 20
    elif word_count >= 80: score += 10

    return {"ats_score": min(score, 100), "skills_found": skill_count, "word_count": word_count}

# Score both versions
original_score = calculate_ats_score_simple(resume_text)
crafted_score = calculate_ats_score_simple(crafted_resume)

print("=== BEFORE vs AFTER COMPARISON ===\n")
print(f"ORIGINAL RESUME:")
print(f"  ATS Score: {original_score['ats_score']}/100")
print(f"  Skills detected: {original_score['skills_found']}")
print(f"  Word count: {original_score['word_count']}\n")

print(f"AI-CRAFTED RESUME:")
print(f"  ATS Score: {crafted_score['ats_score']}/100")
print(f"  Skills detected: {crafted_score['skills_found']}")
print(f"  Word count: {crafted_score['word_count']}\n")

improvement = crafted_score['ats_score'] - original_score['ats_score']
print(f"IMPROVEMENT: {'+' if improvement >= 0 else ''}{improvement} points")

=== BEFORE vs AFTER COMPARISON ===

ORIGINAL RESUME:
  ATS Score: 95/100
  Skills detected: 15
  Word count: 307

AI-CRAFTED RESUME:
  ATS Score: 95/100
  Skills detected: 19
  Word count: 390

IMPROVEMENT: +0 points


In [6]:
comparison_result = {
    "original": original_score,
    "crafted": crafted_score,
    "ats_score_improvement": crafted_score['ats_score'] - original_score['ats_score']
}

with open("../outputs/resume_before_after_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison_result, f, indent=2)

print("Comparison saved to ../outputs/resume_before_after_comparison.json")
print("Notebook 14 (Resume Crafting Agent) — COMPLETE with before/after proof")

Comparison saved to ../outputs/resume_before_after_comparison.json
Notebook 14 (Resume Crafting Agent) — COMPLETE with before/after proof
